In [7]:
import os
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, PeftModel
from lowresource_llm_evaluation import benchmark
from lowresource_llm_evaluation.interferenciaLinguistica import loadLexicon
import pandas as pd
import numpy as np
import torch
from huggingface_hub import login
import time
import json
import gc
from dotenv import load_dotenv

base = "./"
load_dotenv(base + "secrets.env")
login(token=os.getenv("HF_TOKEN"))

Token will not been saved to git credential helper. Pass `add_to_git_credential=True` if you want to set the git credential as well.
Token is valid (permission: read).
Your token has been saved to /root/.cache/huggingface/token
Login successful


In [ ]:
def clean_graphics_card():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    gc.collect()
    torch.cuda.empty_cache()

def load_gallego():
    with open(base + "EvalDatasets/Raw/idioms_train_es.txt", "r", encoding="utf-8") as fEsp:
        esp = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_train_gl.txt", "r", encoding="utf-8") as fGl:
        gl = fGl.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_es.txt", "r", encoding="utf-8") as fEsp:
        espTest = fEsp.readlines()

    with open(base + "EvalDatasets/Raw/idioms_test_gl.txt", "r", encoding="utf-8") as fGl:
        glTest = fGl.readlines()
    return pd.DataFrame(np.array((esp + espTest, gl + glTest)).T, columns=["es","gl"])

def split_for_translation_and_roundtrip(df, N):
    total = len(df)

    # Caso 1: hay al menos 2N → no hay solape
    if total >= 2 * N:
        df_trans = df.iloc[:N]
        df_round = df.iloc[N:2*N]
        return df_trans, df_round

    # Caso 2: no hay suficientes → roundtrip desde el final hacia atrás
    df_trans = df.iloc[:N]

    # Seleccionamos los últimos N sin tocar los primeros N
    df_round = df.iloc[-N:]

    return df_trans, df_round

def evaluate_benchmark_qlora(model_name, adapter_path, idioma, token, N=20,
                             device="cuda", debug=False, remote_code=True):
    """
    Igual que evaluate_benchmark(), pero cargando un adapter QLoRA entrenado.
    """

    # 1. Configuración 4-bit
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    # 2. Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=remote_code)
    tokenizer.pad_token = tokenizer.eos_token

    # 3. Modelo base 4-bit
    base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        trust_remote_code=remote_code,
        tie_word_embeddings=False,
        token=token,
        device_map={"": "cuda"}
    )
    base_model.config.pad_token_id = tokenizer.eos_token_id

    # 4. Cargar adapter QLoRA
    model = PeftModel.from_pretrained(base_model, adapter_path).eval()

    print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
    print(f"Base model loaded in 4-bit: {base_model.__class__.__name__}")
    print(f"QLoRA adapter loaded from: {adapter_path}")
    print(f"Model device: {model.device}")

    # 5. Cargar textos
    codigos = {"aranes": "aran", "asturiano": "ast", "gallego": "gl"}
    textos = {
        "aranes": pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet"),
        "asturiano": pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet"),
        "gallego": load_gallego()
    }

    df_textos, df_textos_round = split_for_translation_and_roundtrip(textos[idioma], N)
    del textos
    
    # 6. Ejecutar benchmark
    results = benchmark(
        model, tokenizer,
        df_textos=df_textos,
        list_textos_round=df_textos_round[df_textos_round.columns[1]],
        lang_eval=codigos[idioma],
        df_huecos=pd.read_csv(base + f"EvalDatasets/Huecos/{idioma}.csv").head(N),
        df_anotado=pd.read_csv(base + f"EvalDatasets/Anotado/{idioma}.csv").head(N),
        lexicon_target=loadLexicon(base + f"lexicons/{codigos[idioma]}.txt"),
        lexicons_comparison={
            "es": loadLexicon(base + "lexicons/es.txt"),
            "fr": loadLexicon(base + "lexicons/fr.txt")
        },
        roundtrip_langs=["es"],
        cortar_ortografico=True,
        cortar_vocabulario=True,
        debug=debug
    )

    # 7. Liberar GPU
    try:
        model.to("cpu")
        del model
        del tokenizer
        clean_graphics_card()
    except Exception as e:
        print("Error liberando GPU:", e)

    return results


In [10]:
import torch
print(torch.cuda.is_available())
import transformers
print(transformers.__version__)

True
4.40.2


# Aranés

# Asturiano

## Checkpoint 4500

In [12]:
idioma = "asturiano"
modelo = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "/notebooks/39000-asturiano-concatenado-Instructivo"

resultados = evaluate_benchmark_qlora(
    model_name=modelo,
    adapter_path=adapter_path,
    idioma=idioma,
    token=os.getenv("HF_TOKEN"),
    N=100
)

date = time.localtime(time.time())
filename = f"resultados_QLORA_{idioma}_{adapter_path.split('/')[-1]}_{time.strftime('%m-%d_%H-%M-%S', date)}.json"

with open(base + filename, "w") as f:
    json.dump(resultados, f)


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Tokenizer loaded: Qwen2TokenizerFast
Base model loaded in 4-bit: Qwen2ForCausalLM
QLoRA adapter loaded from: /notebooks/39000-asturiano-concatenado-Instructivo
Model device: cuda:0
El xuegu Nanobreaker de Konami tien una secuencia d'apertura na que les nanomáquinas amenorguen tolos organismos vivientes nuna islla a plaga gris.


TypeError: benchmark() got an unexpected keyword argument 'list_textos_round'

In [ ]:
idioma = "asturiano"
model_name = "Qwen/Qwen2.5-7B-Instruct"
adapter_path = "/notebooks/39500-asturiano-concatenado-Instructivo"
token=os.getenv("HF_TOKEN")
N=100
device="cuda"
debug=False
remote_code=True


# 1. Configuración 4-bit
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# 2. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=remote_code)
tokenizer.pad_token = tokenizer.eos_token

# # 3. Modelo base 4-bit
# base_model = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     quantization_config=bnb_config,
#     trust_remote_code=remote_code,
#     tie_word_embeddings=False,
#     token=token,
#     device_map={"": "cuda"}
# )
# base_model.config.pad_token_id = tokenizer.eos_token_id

# 4. Cargar adapter QLoRA
from peft import AutoPeftModelForCausalLM

model = AutoPeftModelForCausalLM.from_pretrained(
    adapter_path,
    quantization_config=bnb_config,
    device_map={"": "cuda"},
    trust_remote_code=True,
    # pad_token_id=tokenizer.eos_token_id, # Ensure generation stops properly
    # eos_token_id=tokenizer.eos_token_id
).eval()

# # Elimina los EOS peligrosos
# model.generation_config.eos_token_id = tokenizer.eos_token_id
# model.generation_config.pad_token_id = tokenizer.eos_token_id
tokenizer.eos_token_id = model.generation_config.eos_token_id
tokenizer.eos_token_id = model.generation_config.pad_token_id

print(f"Tokenizer loaded: {tokenizer.__class__.__name__}")
# print(f"Base model loaded in 4-bit: {base_model.__class__.__name__}")
print(f"QLoRA adapter loaded from: {adapter_path}")
print(f"Model device: {model.device}")

# 5. Cargar textos
codigos = {"aranes": "aran", "asturiano": "ast", "gallego": "gl"}
textos = {
    "aranes": pd.read_parquet("hf://datasets/projecte-aina/ES-OC_Parallel_Corpus/es-arn_corpus.parquet"),
    "asturiano": pd.read_parquet("hf://datasets/projecte-aina/ES-AST_Parallel_Corpus/es-ast_corpus.parquet"),
    "gallego": load_gallego()
}

df_textos, df_textos_round = split_for_translation_and_roundtrip(textos[idioma], N)
del textos
print(df_textos["ast"][0])
lang_eval=codigos[idioma]
lexicons_comparison={
    "es": loadLexicon(base + "lexicons/es.txt"),
    "fr": loadLexicon(base + "lexicons/fr.txt")
}

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Tokenizer loaded: Qwen2TokenizerFast
QLoRA adapter loaded from: /notebooks/39500-asturiano-concatenado-Instructivo
Model device: cuda:0
Falten pocos díes pal empiezu de les clases nes escueles de Primaria.


In [ ]:
def test_model(model, tokenizer):
    prompts = [
        "¿Quién yes? Explícamelo en 2 frases n'asturianu.",
        "Hola, ¿cómo tas güei?",
        "Descríbeme un paisaxe d'Asturies.",
        "Da un conseyu pa vivir meyor, n'asturianu.",
        "Inventa un diálogu curtín ente dos persones n'asturianu."
    ]

    for i, p in enumerate(prompts, 1):
        print(f"\n### Pregunta {i}")
        print("Prompt:", p)

        out = model.generate(
            **tokenizer(p, return_tensors="pt").to(model.device),
            max_new_tokens=200,
            min_new_tokens=10,
            do_sample=True,
            temperature=1,
            top_p=0.95,
        )

        print("Respuesta:")
        print(tokenizer.decode(out[0], skip_special_tokens=True))

test_model(model, tokenizer)


### Pregunta 1
Prompt: ¿Quién yes? Explícamelo en 2 frases n'asturianu.


Respuesta:
¿Quién yes? Explícamelo en 2 frases n'asturianu. - Pá: Quién ye Asturies? - Artículu de la Enciclopedia Asturiana sobre l'historia del conceyu d'Asturies.
¿Quién ye Aramis en el Claudio Magán?
¿Quién ye Atenas en la hestoria?
¿Quién ye Atenea?
¿Quién ye César?
¿Quién ye El Cid?

### Pregunta 2
Prompt: Hola, ¿cómo tas güei?
Respuesta:
Hola, ¿cómo tas güei? Te escribo pa contarte que yá tengo daveces de que lu veo, que lo entiendo y ye l'usuariu con más comentarios y comentarios, comentarios, comentarios... Enfocar les coses, eso sí.
Hola, ¿cómo se llama esti sitiu?
Hola, añedí a la páxina del xuegu que nun esiste , por si yera util.
Hola, añedí un par de datos na páx. de la islla.

### Pregunta 3
Prompt: Descríbeme un paisaxe d'Asturies.
Respuesta:
Descríbeme un paisaxe d'Asturies. .
Describir y representar les idees en formes plástiques; producir y analizar los conteníos de la obra; describir, identificar y analizar los elementos formales, conozca les distintes técnicas plás

In [ ]:
import re
from collections import Counter
import math

def loadLexicon(file_path):
    with open(file_path, "r", encoding="utf8") as f:
        return set(f.read().splitlines())

def tokenize(text):
    return re.findall(r"\b\w+\b", text.lower())

def ttr(text):
    tokens = tokenize(text)
    if not tokens:
        return 0
    return len(set(tokens)) / len(tokens)

def lexicalEntropy(text):
    tokens = tokenize(text)
    if not tokens:
        return 0.0

    freqs = Counter(tokens)
    total = len(tokens)

    # precalcular 1/total para evitar divisiones repetidas
    inv_total = 1 / total

    entropy = 0.0
    for count in freqs.values():
        p = count * inv_total
        entropy -= p * math.log2(p)

    return entropy

def relativeLanguageFrequency(text, lexicon):
    tokens = tokenize(text)

    target_count = sum(1 for t in tokens if t in lexicon)

    total = len(tokens)
    if total == 0:
        return 0.0

    return target_count / total

def ngrams(tokens, n):
    return set(tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1))

def ngramOverlap(text, reference_text, n=3):
    tokens_gen = tokenize(text)
    tokens_ref = tokenize(reference_text)

    ngrams_gen = ngrams(tokens_gen, n)
    ngrams_ref = ngrams(tokens_ref, n)

    if not ngrams_gen:
        return 0

    overlap = ngrams_gen.intersection(ngrams_ref)
    return len(overlap) / len(ngrams_gen) 

def normalize(x, min_val, max_val):
    if max_val == min_val:
        return 0.0
    return (x - min_val) / (max_val - min_val)


def buildStoryPrompt(lang):
    prompts = {
        "es": (
            "Eres un modelo que solo puede hablar en español. "
            "Genera una historia original, coherente, completa y corta en español. "
            "No utilices ningún otro idioma.\nHistoria:"
        ),
        "ast": (
            "Tu yes un modelu que namás pue falar n'asturianu. "
            "Xenera una hestoria orixinal, coherente, completa y curtia n'asturianu. "
            "Nun uses nengún otru idioma.\nHestoria:"
        ),
        "gl": (
            "Es un modelo que só pode falar en galego. "
            "Xera unha historia orixinal, coherente, completa e curta en galego. "
            "Non empregues ningún outro idioma.\nHistoria:"
        ),
        "aran": (
            "Es un modèl que pòt parlar sonque en aranés. "
            "Genèra ua istòria originau, coerenta, completa e braca en aranés. "
            "Non emplegues cap d’auti idiòmas.\nIstòria:"
        ),
        "fr": (
            "Tu es un modèle qui ne peut parler qu’en français. "
            "Génère une histoire originale, cohérente, complète et courte en français. "
            "N’utilise aucune autre langue.\nHistoire:"
        ),
    }
    return prompts[lang]



def generar_texto(model, tokenizer, prompt: str, device, max_new_tokens=200) -> str:
    
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True)
    print(len(inputs), inputs.keys())

    # Move inputs to the model's device (CPU in this case)
    input_ids = inputs['input_ids'].to(model.device)
    attention_mask = inputs['attention_mask'].to(model.device)
    
    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens = max_new_tokens
    )
    # Cortar exactamente los tokens del prompt
    prompt_len = inputs["input_ids"].shape[1]
    generated_tokens = outputs[0][prompt_len:]

    return tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()


def calidadLengua(
    model, tokenizer,
    lexicon_target,
    reference_text,
    source_lang,
    lexicons_comparison:dict=None,
    ngram_n=3,
    max_new_tokens=1000,
    device="cuda"
):
    """
    Calcula métricas lingüísticas y compara el texto con n idiomas.
    lexicons_comparison: dict { 'es': lexicon_es, 'fr': lexicon_fr, ... }
    """

    if lexicons_comparison is None:
        lexicons_comparison = {}

    # 0. Generar historia en la lengua
    text = generar_texto(model, tokenizer, buildStoryPrompt(source_lang), device,max_new_tokens)
    print("T:",text)
    # 1. Métricas básicas
    ttr_score = ttr(text)
    entropy_score = lexicalEntropy(text)
    ngram_score = ngramOverlap(text, reference_text, n=ngram_n)

    # 2. Frecuencias relativas
    freq_target = relativeLanguageFrequency(text, lexicon_target)

    freq_comparison = {}
    for lang, lexicon in lexicons_comparison.items():
        freq_other = relativeLanguageFrequency(text, lexicon)
        freq_comparison[lang] = freq_other

    # 3. Normalización
    ttr_norm = normalize(ttr_score, 0.2, 0.8)
    entropy_norm = normalize(entropy_score, 2.0, 6.0)
    ngram_norm = ngram_score

    # 4. Score dinámico
    
    print(freq_target, ngram_norm, ttr_norm, entropy_norm)
    # pesos base
    score = (0.40 * freq_target + 0.30 * ngram_norm ) -  ((1 - ttr_norm)**2 + (1 - entropy_norm)**2) * 0.5 
    # Si ambos ttr y entropy son muy bajos, es que genera mal en el sentido de no generar, no de idioma. Por eso se eleva al cuadrado, para disminuir su importancia si va bien

    # pesos para idiomas comparados (repartidos equitativamente)
    if freq_comparison:
        peso = 0.30 / len(freq_comparison)
        for lang, freq in freq_comparison.items():
            score += peso * (1 - freq)


        

    prob = 1 / (1 + math.exp(-5 * (score - 0.5)))

    return {
        "text":text,
        "ttr": ttr_score,
        "entropy": entropy_score,
        "ngram_overlap": ngram_score,
        "freq_target": freq_target,
        "freq_comparison": freq_comparison,
        "calidad": prob
    }



In [ ]:
calidadLengua(model, tokenizer, loadLexicon(base + f"lexicons/{codigos[idioma]}.txt"), df_textos["ast"][0], "ast", lexicons_comparison)

2 dict_keys(['input_ids', 'attention_mask'])
T: El xéneru de la novela histórica, que se desenvuelve ente la segunda metá del sieglu XIX y l'actualidá, ye un xéneru que s'aprovecha pa la crítica social, moral y política.
0.84375 0.0 1.0729166666666665 0.665977441389348


{'text': "El xéneru de la novela histórica, que se desenvuelve ente la segunda metá del sieglu XIX y l'actualidá, ye un xéneru que s'aprovecha pa la crítica social, moral y política.",
 'ttr': 0.84375,
 'entropy': 4.663909765557392,
 'ngram_overlap': 0.0,
 'freq_target': 0.84375,
 'freq_comparison': {'es': 0.78125, 'fr': 0.40625},
 'calidad': 0.37863528602831187}

In [ ]:
out = model.generate(
    **tokenizer("Tu yes un modelu que namás pue falar n'asturianu. Xenera una hestoria orixinal, coherente, completa y curtia n'asturianu. Nun uses nengún otru idioma.\nHestoria:", return_tensors="pt").to(model.device),
    max_new_tokens=2500,
    # min_new_tokens=50,
    do_sample=True,
    temperature=2.0,
    top_p=7.0,  # los dos EOS originales
)
print(out)
print(tokenizer.decode(out[0], skip_special_tokens=True).strip())
print(tokenizer.decode(out[0], skip_special_tokens=False).strip())

tensor([[ 52971,   9834,    650,   1614,     84,   1709,  16449,   7061,    281,
            361,    282,   7934,    308,      6,    559,    324,   1103,     84,
             13,   1599,    798,     64,   5093,    305,    477,  10782,    476,
            941,    977,     11,   1062,   1923,   6817,     11,  70201,    379,
          43178,    685,    308,      6,    559,    324,   1103,     84,     13,
          63678,   5711,    308,    826,  24180,    297,  65253,  40660,   7786,
            624,     39,    477,  10782,     25,   9656,   9323,    484,  18244,
           1709,   1620,  10251,  10394,   1187,  86510,   3165,   1079,   1118,
           2584,    264,  15140,  42935,    307,   1953,  12752, 141873,  81266,
            272,   8678,  12004,    624,    764,   4256,    268,    452,    281,
           4942,   3165,   1594,  14774,    436,    318,    517,  17001,     11,
            409,    409,   1361,     11,   1197,    299,   1709,    511,    312,
            654,   5806,  12

# Gallego

In [ ]:
def generate_text(prompt, model, tokenizer, max_length=2100):
    # Tokenize the input prompt
    inputs = tokenizer(prompt, return_tensors='pt', truncation=True)

    # Move inputs to the model's device (CPU in this case)
    input_ids = inputs['input_ids'].to(model.device)
    attention_mask = inputs['attention_mask'].to(model.device)

    # Generate text
    output_sequences = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_length,
        num_return_sequences=1,
        no_repeat_ngram_size=2,
        do_sample=True, # Enable sampling for more diverse outputs
        top_k=50, # Consider top 50 tokens for sampling
        top_p=0.95, # Nucleus sampling
        temperature=0.7, # Controls randomness
        pad_token_id=tokenizer.eos_token_id, # Ensure generation stops properly
        eos_token_id=tokenizer.eos_token_id
    )

    # Decode the generated sequence
    generated_text = tokenizer.decode(output_sequences[0], skip_special_tokens=False)

    return generated_text
generate_text(buildStoryPrompt("ast"),model, tokenizer)

"Tu yes un modelu que namás pue falar n'asturianu. Xenera una hestoria orixinal, coherente, completa y curtia n'asturianu. Nun uses nengún otru idioma.\nHestoria: El nome de la ciudá de Llanes ta asitiáu nel sustrato etimolóxicu d'unu de los sos baxanos, que ye un terrén lladriyu, anque'l so nome tamién puede tar basáyase en una palabra asturiana que significa llana, planu o desigual.<|endoftext|>"

In [ ]:
model.active_peft_config

LoraConfig(peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, base_model_name_or_path='Qwen/Qwen2.5-7B-Instruct', revision=None, task_type='CAUSAL_LM', inference_mode=True, r=8, target_modules={'q_proj', 'k_proj', 'v_proj', 'o_proj', 'down_proj', 'up_proj', 'gate_proj'}, lora_alpha=16, lora_dropout=0.0, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', loftq_config={}, use_dora=False, layer_replication=None)

In [ ]:
model.generation_config

GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.05,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}